# 0. 원천 데이터 → els3_dataset 생성 (base)

raw DART(`AUTO_CALL`/`SCHD_INFO`/`UDLY_INFO`) + `data/cache`(기초자산 일별종가 `px_*`, KRW 금리곡선 `krw_curve`)로부터 **3-star STEP KRW ELS**의 학습 base 데이터셋을 직접 재유도해 `data/els3_dataset.parquet` 로 저장한다.

- **branch(시장상태)**: 바스켓 변동성 `sig1..3`/`sig_eff` (180일 역사), 상관 `rho12/13/23` (180일 역사), KRW Nelson-Siegel 수익률곡선 `u0..u9`/`r`/`curve_*`.
- **계약/보조**: `B`/`coupon`/`tenor`/`nobs`/`cpn_spread`/`b_over_k`/`stepdown`/`mom6m` + 발행메타(issuer/risk/…), `fair`(공정가치), 인과적 `recent_mktvol`.

**이 단계엔 MC 이론가를 계산하지 않는다** — `mc`·`recent_margin` 은 `1_MC_recompute` 에서 추가하며, 학습 CSV 2종(`ml`/`deeponet`)도 mc 가 붙은 뒤 거기서 생성한다. (이 노트북은 raw→base 검증용)

In [1]:
# ===== raw + cache → els3_dataset base 재유도 =====
# module.build_source: exp15 로직을 현행 정의(180일 역사 vol/corr, KRW NS 공섬)로 이식. MC 미포함.
from module import build_source
df = build_source.build_source()
print("\ncolumns:", len(df.columns))
print(list(df.columns))

4구조 후보: 60358
   {('LIZARD', 0): 10293, ('LIZARD', 1): 1873, ('STEP', 0): 24574, ('STEP', 1): 23618}


built 58790 products in 236s
  구조별: {('LIZARD', 0): 9880, ('LIZARD', 1): 1781, ('STEP', 0): 24048, ('STEP', 1): 23081}
  탈락 사유: {'티커 매핑 실패': 906, '스케줄: pmt_na': 161, '리자드 FROM_PREV': 126, '가격이력 없음': 124, '180일 변동성 부족': 123, '스케줄: nobs>12': 112, '스케줄: strk_na': 15, '스케줄: nobs<2': 1}


saved C:\Users\Juhwankim\Documents\Github\PI-DeepONet-ELS\data\els3_dataset.parquet rows 58790

columns: 106
['item', 'sig1', 'sig2', 'sig3', 'rho12', 'rho13', 'rho23', 'sig_mean', 'sig_max', 'sig_min', 'rho', 'sig_eff', 'cpn_spread', 'b_over_k', 'stepdown', 'mom6m', 'isu_ord', 'r', 'B', 'Kfirst', 'K', 'coupon', 'tenor', 'nobs', 'fair', 'issuer', 'risk', 'ptype', 'rdmp', 'imonth', 'amt', 'sbrt', 'dvrt', 'prcp', 'kigrc', 'iyear', 'subdays', 'u0', 'u1', 'u2', 'u3', 'u4', 'u5', 'u6', 'u7', 'u8', 'u9', 'curve_level', 'curve_slope', 'curve_curv', 'opt_type', 'ki_yn', 'udl1', 'udl2', 'udl3', 'udl_key', 'strk_0', 'strk_1', 'strk_2', 'strk_3', 'strk_4', 'strk_5', 'strk_6', 'strk_7', 'strk_8', 'strk_9', 'strk_10', 'strk_11', 'pmt_0', 'pmt_1', 'pmt_2', 'pmt_3', 'pmt_4', 'pmt_5', 'pmt_6', 'pmt_7', 'pmt_8', 'pmt_9', 'pmt_10', 'pmt_11', 'lz_barr_0', 'lz_barr_1', 'lz_barr_2', 'lz_barr_3', 'lz_barr_4', 'lz_barr_5', 'lz_barr_6', 'lz_barr_7', 'lz_barr_8', 'lz_barr_9', 'lz_barr_10', 'lz_barr_11', 'lz_pm

## base 검증

MC 전 base 상태 확인: branch 피처 완전성(NaN 0), 범위, 발행기간. `mc`·`recent_margin` 은 아직 없어야 정상(1_MC_recompute에서 추가).

In [2]:
import pandas as pd, numpy as np
from util import file_manager as fm

d = pd.read_parquet(fm.source())
branch = ["sig1", "sig2", "sig3", "rho12", "rho13", "rho23", "sig_eff",
          "u0", "u9", "r", "curve_level", "curve_slope", "curve_curv", "recent_mktvol"]
print(f"source -> {fm.source().relative_to(fm.ROOT)} | rows {len(d)} | cols {len(d.columns)}")
print(f"branch 피처 present: {all(c in d.columns for c in branch)} | NaN: {int(d[branch + ['fair']].isna().sum().sum())}")
print(f"has mc? {'mc' in d.columns} | has recent_margin? {'recent_margin' in d.columns}  (둘 다 False 가 정상)")
print(f"fair [{d.fair.min():.3f}, {d.fair.max():.3f}] | tenor [{d.tenor.min():.2f}, {d.tenor.max():.2f}] | "
      f"sig_eff 평균 {d.sig_eff.mean():.3f}")
yr = pd.to_datetime(d.isu_ord.map(lambda o: pd.Timestamp.fromordinal(int(o))))
print(f"발행기간 {yr.min().date()} ~ {yr.max().date()}")
display(d[branch + ["fair", "coupon", "tenor"]].describe().round(4))

source -> data\els3_dataset.parquet | rows 58790 | cols 106
branch 피처 present: True | NaN: 0
has mc? False | has recent_margin? False  (둘 다 False 가 정상)
fair [0.700, 1.050] | tenor [0.99, 5.00] | sig_eff 평균 0.236
발행기간 2015-01-05 ~ 2026-05-12


,sig1,sig2,sig3,rho12,rho13,rho23,sig_eff,u0,u9,r,curve_level,curve_slope,curve_curv,recent_mktvol,fair,coupon,tenor
count,58790.0000,58790.0000,58790.0000,58790.0000,58790.0000,58790.0000,58790.0000,58790.0000,58790.0000,58790.0000,58790.0000,58790.0000,58790.0000,58790.0000,58790.0000,58790.0000,58790.0000
mean,0.1482,0.1799,0.2256,0.1992,0.3092,0.5413,0.2365,0.0193,0.0244,0.0286,0.0256,0.0051,0.0122,0.1840,0.9293,0.0576,2.9945
std,0.0434,0.0540,0.0694,0.1223,0.1175,0.1137,0.0635,0.0094,0.0074,0.0108,0.0095,0.0054,0.0107,0.0455,0.0468,0.0216,0.1431
min,0.0662,0.0958,0.1022,-0.1556,-0.0987,0.0566,0.1165,0.0063,0.0125,0.0110,0.0124,-0.0058,-0.0058,0.1051,0.7002,0.0000,0.9884
25%,0.1232,0.1404,0.1797,0.1145,0.2308,0.4753,0.1946,0.0139,0.0185,0.0223,0.0191,0.0008,0.0067,0.1597,0.8998,0.0429,2.9925
50%,0.1361,0.1676,0.2152,0.1999,0.2941,0.5414,0.2241,0.0165,0.0229,0.0258,0.0229,0.0053,0.0092,0.1721,0.9329,0.0542,2.9979
75%,0.1679,0.2068,0.2543,0.2783,0.3838,0.6062,0.2669,0.0251,0.0284,0.0309,0.0293,0.0092,0.0158,0.2051,0.9638,0.0700,3.0007
max,0.4034,0.5522,0.6983,0.7705,0.8123,0.9990,0.6858,0.0402,0.0427,0.0755,0.0604,0.0165,0.0733,0.3446,1.0497,0.3000,4.9993
